# Lecture 3 — Class Exercise
## Line Charts: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
- No spaghetti — if you show multiple lines, grey everything and highlight one
- Remove chart borders, heavy gridlines, rotating text
- Label directly; avoid legends where possible
- Insight title


In [5]:
%pip install plotly nbformat
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [10]:
import os, pandas as pd

print("Working dir:", os.getcwd())

found = None
for root, dirs, files in os.walk('/'):
    if root.startswith(('/proc', '/sys', '/lib', '/usr')):
        continue
    if 'co2_emissions.csv' in files:
        found = os.path.join(root, 'co2_emissions.csv')
        break

print("Found at:", found)
df = pd.read_csv(found)
print(df.shape)

Working dir: /drive
Found at: /drive/notebooks/co2_emissions.csv
(345, 5)


In [11]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: <StringArray>
[ 'United States',          'China',          'India',        'Germany',
 'United Kingdom',         'France',         'Brazil',          'Japan',
         'Canada',      'Australia',    'South Korea',         'Russia',
   'South Africa',         'Mexico',      'Indonesia']
Length: 15, dtype: str

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#EEEEEE), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

In [14]:
%pip install plotly nbformat

In [15]:
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)

# Finds the CSV wherever the kernel put it
found = None
for root, dirs, files in os.walk('/'):
    if root.startswith(('/proc', '/sys', '/lib', '/usr')):
        continue
    if 'co2_emissions.csv' in files:
        found = os.path.join(root, 'co2_emissions.csv')
        break

df = pd.read_csv(found)
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())

Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [16]:
# Task — Multi-series line with highlight

from IPython.display import HTML, display

HIGHLIGHT = 'China'          # the story we want to tell
HL_COLOR  = '#C0392B'        # highlight colour
GREY      = '#EEEEEE'        # context lines

# 1. Filter: all Asian countries
asia = df[df['Region'] == 'Asia'].sort_values(['Country', 'Year'])
print("Asian countries:", sorted(asia['Country'].unique()))

# 2. Numbers for the insight title (computed, not hard-coded)
hl = asia[asia['Country'] == HIGHLIGHT].set_index('Year')['CO2_Mt']
y0, y1 = hl.index.min(), hl.index.max()
growth = hl[y1] / hl[y0]
title = f"{HIGHLIGHT}'s CO2 emissions grew {growth:.1f}x since {y0}, dwarfing every other Asian economy"

# 3. Build the chart: grey context lines first, highlight drawn last (on top)
fig = go.Figure()

for country, g in asia.groupby('Country'):
    if country == HIGHLIGHT:
        continue
    fig.add_trace(go.Scatter(
        x=g['Year'], y=g['CO2_Mt'], mode='lines',
        line=dict(color=GREY, width=1.5),
        hoverinfo='skip', showlegend=False
    ))
    fig.add_annotation(
        x=g['Year'].max(), y=g['CO2_Mt'].iloc[-1],
        text=country, showarrow=False, xanchor='left', xshift=6,
        font=dict(size=10, color='#AAAAAA')
    )

fig.add_trace(go.Scatter(
    x=hl.index, y=hl.values, mode='lines',
    line=dict(color=HL_COLOR, width=4),
    hovertemplate='%{x}: %{y:,.0f} Mt<extra></extra>',
    showlegend=False
))

# 4. Direct label at the end of the highlighted line (no legend)
fig.add_annotation(
    x=y1, y=hl[y1],
    text=f"<b>{HIGHLIGHT}</b><br>{hl[y1]:,.0f} Mt",
    showarrow=False, xanchor='left', xshift=8,
    font=dict(size=13, color=HL_COLOR)
)

# 5. Clean styling: no borders, light gridlines, horizontal text
fig.update_layout(
    title=dict(text=title, x=0, xanchor='left', font=dict(size=20, color='#222222')),
    template='plotly_white',
    plot_bgcolor='white',
    showlegend=False,
    width=950, height=550,
    margin=dict(l=70, r=130, t=90, b=60),
    xaxis=dict(showgrid=False, showline=False, zeroline=False,
               tickangle=0, dtick=2, title=None),
    yaxis=dict(showgrid=True, gridcolor='#F2F2F2', gridwidth=0.5,
               showline=False, zeroline=False,
               title=dict(text='CO2 emissions (Mt)', font=dict(size=12, color='#666666')),
               tickformat=','),
    font=dict(color='#444444')
)
fig.add_annotation(
    text="Source: Our World in Data", xref='paper', yref='paper',
    x=0, y=-0.12, showarrow=False, xanchor='left',
    font=dict(size=10, color='#999999')
)

# 6. Display (works in browser Jupyter)
display(HTML(fig.to_html(include_plotlyjs='cdn', full_html=False)))

Asian countries: ['China', 'India', 'Indonesia', 'Japan', 'South Korea']
